In [9]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse


from model.dataset import BagsDataset, custom_collate_fn
from  model.model import MIL, EarlyStopping
from model.dataset import BagsDataset, custom_collate_fn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
all_genes = pd.read_csv('human.csv')
all_genes = all_genes['Gene'].values
dataset = BagsDataset('training.csv', immune_cell='tcell')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Processing: adata=HumanLungCancerFFPE.h5ad, radius=200, resolution=low
Processing: adata=UKF275.h5ad, radius=200, resolution=high
Processing: adata=TumE2.h5ad, radius=200, resolution=high
Processing: adata=TumA1.h5ad, radius=200, resolution=high


Creating Bags with radius 200: 100%|██████████████████████████| 2640/2640 [00:00<00:00, 7350.28it/s]


Total bags created: 13450
Average instances per bag: 9


In [4]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)



In [5]:
model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [6]:
num_epochs = 1
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.0001)

In [7]:
ig_scores_before_training = torch.sigmoid(model.immunogenicity.ig)
with open('ig_scores_before_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training'])
    for gene, score in zip(all_genes, ig_scores_before_training):
        writer.writerow([gene, score.item()])

In [8]:
for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)
            
            output = model(distances, gene_expressions, list(current_genes[0]))
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')
    

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_current_genes in val_dataloader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_current_genes[0]))
            val_loss += criterion(val_output, val_label).item()
    
    val_loss /= len(val_dataloader)
    print(f'Validation Loss: {val_loss:.4f}')

    early_stopping(val_loss, model, epoch)
    if early_stopping.early_stop:
        print(f'Early stopping at epoch {epoch+1}')
        break


Epoch 1/1: 100%|██████████| 9415/9415 [00:26<00:00, 358.39batch/s, loss=0.566]


Epoch [1/1], Loss: 0.6328
Validation Loss: 0.6259


In [ ]:
ig_scores_after_training = torch.sigmoid(model.immunogenicity.ig)
with open('ig_scores_after_training.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score After Training'])
    for gene, score in zip(all_genes, ig_scores_after_training):
        writer.writerow([gene, score.item()])
        
with open('ig_score_changes.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])